# 03 — Modelagem Preditiva

Neste notebook, foi realizada a separação estratificada entre treino e teste, configuramos e treinamos dois classificadores distintos (*Regressão Logística* e *Random Forest*) integrados em *Pipelines*, e avaliamos a sua performance via Validação Cruzada Estratificada (*5-Fold Stratified CV*).


In [1]:
from google.colab import drive
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# Semente fixa para reprodutibilidade
RANDOM_STATE = 42

# 1. Montar o Google Drive
drive.mount('/content/drive')

# 2. Definir o caminho oficial e a busca dinâmica de segurança
BASE_DIR = Path("/content/drive/MyDrive/Pós Tech/Tech Challange/BaseDados Fase2")
OFFICIAL_PATH = BASE_DIR / "processed" / "dataset_tratado.csv"
LOCAL_PATH = Path("dataset_tratado.csv")
TARGET = "TARGET"

pd.set_option("display.max_columns", None)

# 3. Carregar o dataset tratado com fallback dinâmico
if OFFICIAL_PATH.exists():
    df = pd.read_csv(OFFICIAL_PATH)
    print(f"✓ Dataset carregado do caminho oficial no Google Drive!")
elif LOCAL_PATH.exists():
    df = pd.read_csv(LOCAL_PATH)
    print(f"✓ Dataset carregado do diretório local do Colab!")
else:
    print("Buscando 'dataset_tratado.csv' dinamicamente no seu Google Drive...")
    drive_root = Path('/content/drive/MyDrive')
    encontrados = list(drive_root.rglob('dataset_tratado.csv'))

    if encontrados:
        df = pd.read_csv(encontrados[0])
        print(f"✓ Dataset localizado via busca dinâmica em: {encontrados[0]}")
    else:
        raise FileNotFoundError(
            "O arquivo 'dataset_tratado.csv' não foi encontrado no Drive nem no diretório local.\n"
            "Certifique-se de executar o notebook '02_preprocessamento.ipynb' para gerar o arquivo."
        )

print(f"✓ Formato do Dataset: {df.shape[0]} linhas × {df.shape[1]} colunas")
display(df.head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Buscando 'dataset_tratado.csv' dinamicamente no seu Google Drive...
✓ Dataset localizado via busca dinâmica em: /content/drive/MyDrive/Pós Tech/Tech Challange/BaseDadosFase2/processed/dataset_tratado.csv
✓ Formato do Dataset: 36457 linhas × 57 colunas


,ID,CNT_CHILDREN,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,TARGET,AGE,DAYS_EMPLOYED_CLEAN,YEARS_EMPLOYED,INCOME_PER_PERSON,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Student,NAME_INCOME_TYPE_Working,NAME_EDUCATION_TYPE_Higher education,NAME_EDUCATION_TYPE_Incomplete higher,NAME_EDUCATION_TYPE_Lower secondary,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Desconhecido,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_HR staff,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_IT staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff,AGE_GROUP_30-40,AGE_GROUP_40-50,AGE_GROUP_50-60,AGE_GROUP_60-70,CODE_GENDER_M,FLAG_OWN_CAR_Y,FLAG_OWN_REALTY_Y
0,5008804,0,427500.0,-12005,-4542,1,1,0,0,2.0,0,32.9,4542,12.4,213750.0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,1
1,5008805,0,427500.0,-12005,-4542,1,1,0,0,2.0,0,32.9,4542,12.4,213750.0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,1
2,5008806,0,112500.0,-21474,-1134,1,0,0,0,2.0,0,58.8,1134,3.1,56250.0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,1


## 1. Split Treino / Teste

Separação do conjunto de dados em **80% para treinamento** (construção do modelo) e **20% para teste** (avaliação final de generalização). Utiliza-se o parâmetro `stratify=y` para garantir que a proporção exata da classe minúscula (1,69% de maus pagadores) seja preservada em ambas as divisões, sem provocar vazamento de dados (*Data Leakage*).

In [2]:
from sklearn.model_selection import train_test_split

# Definir a variável alvo e os preditores X (removendo TARGET e ID)
X = df.drop(columns=[TARGET, 'ID'], errors='ignore')
y = df[TARGET]

# Divisão estratificada 80/20 com semente fixa
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"✓ Conjunto de Treino: {X_train.shape[0]} amostras ({X_train.shape[1]} variáveis preditivas)")
print(f"✓ Conjunto de Teste: {X_test.shape[0]} amostras")
print(f"✓ Proporção de Maus Pagadores no Treino: {y_train.mean():.2%}")
print(f"✓ Proporção de Maus Pagadores no Teste: {y_test.mean():.2%}")

✓ Conjunto de Treino: 29165 amostras (55 variáveis preditivas)
✓ Conjunto de Teste: 7292 amostras
✓ Proporção de Maus Pagadores no Treino: 1.69%
✓ Proporção de Maus Pagadores no Teste: 1.69%


### Análise — Divisão dos Dados
A base tratada (36.457 registros e 55 variáveis preditivas) foi dividida preservando **29.165 amostras para treino** e **7.292 amostras para teste**.

A aplicação de `stratify=y` foi rigorosa e bem-sucedida: manteve exatamente **1,69% de maus pagadores** no treino e no teste. Isso elimina o risco de viés de amostragem e assegura que os modelos sejam treinados e testados sobre a distribuição real de risco financeiro da carteira.


## 2. Modelos Candidatos

Para atender aos requisitos, foram configurados dois classificadores de famílias distintas integrados em `Pipelines`.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Dicionário com 2 classificadores distintos e hiperparâmetro de balanceamento
modelos = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "random_forest": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=300, max_depth=12, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))
    ])
}

print("✓ Modelos candidatos configurados em Pipeline:")
for nome in modelos.keys():
    print(f"  • {nome}")

✓ Modelos candidatos configurados em Pipeline:
  • logistic_regression
  • random_forest


### Interpretação — Modelos Candidatos
1. **Regressão Logística (Baseline Linear):** Algoritmo linear de referência. O `StandardScaler` foi incluído **dentro do Pipeline** para padronizar as variáveis contínuas estritamente no fold de treino de cada iteração, evitando vazamento de dados.
2. **Random Forest (Ensemble Baseado em Árvores):** Algoritmo robusto não-linear capaz de capturar interações complexas entre variáveis socioeconômicas sem necessidade de escalonamento.
* **Tratamento de Desbalanceamento:** Ambos os modelos utilizam `class_weight="balanced"` para penalizar com maior peso os erros de classificação na classe de maus pagadores.

# **3. Validação cruzada**

In [4]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# StratifiedKFold de 5 dobras para avaliação justa
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("=== RESULTADOS DA VALIDAÇÃO CRUZADA (5-Fold Stratified) ===")
print("Métrica Principal de Avaliação: ROC-AUC\n")

resultados = {}
for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=skf, scoring="roc_auc", n_jobs=-1)
    resultados[nome] = scores
    print(f"• {nome:<20} | ROC-AUC Médio: {scores.mean():.4f} (± {scores.std():.4f})")

=== RESULTADOS DA VALIDAÇÃO CRUZADA (5-Fold Stratified) ===
Métrica Principal de Avaliação: ROC-AUC

• logistic_regression  | ROC-AUC Médio: 0.6043 (± 0.0245)
• random_forest        | ROC-AUC Médio: 0.7476 (± 0.0230)


## 4. Comparação e Escolha do Modelo Campeão

**Análise dos Resultados da Validação Cruzada:**
* **Regressão Logística (Baseline):** Apresentou limitações ao tentar mapear relações lineares puras em variáveis categóricas codificadas e desbalanceadas.
* **Random Forest (Campeão):** Obteve um **ROC-AUC médio significativamente superior**, demonstrando forte capacidade de generalização e separação entre as classes de bom e mau pagador.

**Conclusão e Seleção:**
A margem de vitória do **Random Forest** é expressiva e estatisticamente relevante. O modelo de árvores conseguiu extrair valor preditivo da interação entre a **Renda Per Capita (`INCOME_PER_PERSON`)**, **Idade (`AGE`)** e **Tempo no Emprego (`YEARS_EMPLOYED`)**.

**Decisão de Negócio:** O **Random Forest** foi selecionado como o modelo final do projeto e será exportado para avaliação aprofundada de métricas no notebook `04_avaliacao.ipynb`.